# Fine-Tune Qwen2-VL-2B: LoRA vs QLoRA vs DoRA

Multi-task UI screenshot-to-code generation (HTML, JSX, Vue SFC).

**Requirements:**
- Kaggle GPU T4 x2 (15 GB VRAM)
- Internet enabled (for model download)
- Uploaded datasets: `finetune1_data` (JSONL files) + `websight-images` (PNG images)

**Run one method per session** (12hr Kaggle limit). Set `METHOD` in the **Run Training** cell (e.g. `"qlora"`).

## 1. Install Dependencies

In [ ]:
!pip install -q peft trl qwen-vl-utils bitsandbytes accelerate sacrebleu rouge-score codebleu scikit-image transformers
!nvidia-smi

## 2. Data Setup

### How to upload your data to Kaggle:

**Step 1: Create two Kaggle Datasets**

On your local machine, prepare two zip files:

```
finetune1_data.zip
  train.jsonl
  val.jsonl
  test.jsonl

websight-images.zip
  images/
    000000.png
    000001.png
    ...
```

Then:
1. Go to kaggle.com -> Your Profile -> Datasets -> New Dataset
2. Upload `finetune1_data.zip`, name it `finetune1_data`
3. Upload `websight-images.zip`, name it `websight-images`

**Step 2: Add datasets to this notebook**
1. Click **Add Data** (right sidebar)
2. Search for your datasets and add both
3. They will mount at `/kaggle/input/finetune1_data/` and `/kaggle/input/websight-images/`

**Step 3: Update image paths in JSONL**

The cell below rewrites image paths in the JSONL files to point to the Kaggle mount.

In [ ]:
import os
import json
import glob

# Auto-detect Kaggle dataset paths
KAGGLE_INPUT = "/kaggle/input"
WORK_DIR = "/kaggle/working"

# List everything under /kaggle/input recursively (2 levels)
print("Scanning /kaggle/input...")
for root, dirs, files in os.walk(KAGGLE_INPUT):
    depth = root.replace(KAGGLE_INPUT, '').count(os.sep)
    if depth < 3:
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

# Find JSONL files (search recursively)
jsonl_dir = None
image_dir = None
for root, dirs, files in os.walk(KAGGLE_INPUT):
    if 'train.jsonl' in files:
        jsonl_dir = root
    png_files = [f for f in files if f.endswith('.png')]
    if len(png_files) > 10:
        image_dir = root

assert jsonl_dir, f"Could not find train.jsonl anywhere under {KAGGLE_INPUT}. Upload finetune1_data dataset."
assert image_dir, f"Could not find PNG images under {KAGGLE_INPUT}. Upload websight-images dataset."

print(f"JSONL dir: {jsonl_dir}")
print(f"Image dir: {image_dir}")
print(f"Image count: {len(os.listdir(image_dir))}")

# Rewrite JSONL files with corrected image paths
os.makedirs(os.path.join(WORK_DIR, "data"), exist_ok=True)

for split in ["train", "val", "test"]:
    src = os.path.join(jsonl_dir, f"{split}.jsonl")
    dst = os.path.join(WORK_DIR, "data", f"{split}.jsonl")
    count = 0
    with open(src, "r") as fin, open(dst, "w") as fout:
        for line in fin:
            obj = json.loads(line)
            for msg in obj["messages"]:
                if isinstance(msg["content"], list):
                    for part in msg["content"]:
                        if part.get("type") == "image":
                            old_path = part["image"].replace("file:///", "")
                            fname = os.path.basename(old_path)
                            part["image"] = f"file:///{image_dir}/{fname}"
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            count += 1
    print(f"  {split}.jsonl: {count} examples -> {dst}")

DATA_DIR = os.path.join(WORK_DIR, "data")
print(f"\nData ready at: {DATA_DIR}")

## 3. Configs

In [ ]:
CONFIGS = {
    "lora": {
        "method": "lora",
        "description": "Standard LoRA -- base model in fp16",
        "quantization": None,
        "lora_r": 64,
        "lora_alpha": 128,
        "lora_dropout": 0.05,
        "use_dora": False,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "learning_rate": 1e-4,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "warmup_steps": 100,
        "weight_decay": 0.01,
        "fp16": True,
        "bf16": False,
        "gradient_checkpointing": True,
        "max_seq_length": 2048,
        "logging_steps": 10,
        "save_strategy": "steps",
        "save_steps": 500,
        "eval_strategy": "no",
        "eval_steps": 500,
        "save_total_limit": 1,
        "dataloader_num_workers": 2,
        "per_device_eval_batch_size": 1,
        "seed": 42,
    },
    "qlora": {
        "method": "qlora",
        "description": "QLoRA -- 4-bit NF4 quantized base + LoRA adapters",
        "quantization": {
            "load_in_4bit": True,
            "bnb_4bit_quant_type": "nf4",
            "bnb_4bit_use_double_quant": True,
            "bnb_4bit_compute_dtype": "float16",
        },
        "lora_r": 64,
        "lora_alpha": 128,
        "lora_dropout": 0.05,
        "use_dora": False,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "learning_rate": 1e-4,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "warmup_steps": 100,
        "weight_decay": 0.01,
        "fp16": True,
        "bf16": False,
        "gradient_checkpointing": True,
        "max_seq_length": 2048,
        "logging_steps": 10,
        "save_strategy": "steps",
        "save_steps": 500,
        "eval_strategy": "no",
        "eval_steps": 500,
        "save_total_limit": 1,
        "dataloader_num_workers": 2,
        "per_device_eval_batch_size": 1,
        "seed": 42,
    },
    "dora": {
        "method": "dora",
        "description": "DoRA -- weight-decomposed LoRA (magnitude + direction)",
        "quantization": None,
        "lora_r": 64,
        "lora_alpha": 128,
        "lora_dropout": 0.05,
        "use_dora": True,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "learning_rate": 1e-4,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "warmup_steps": 100,
        "weight_decay": 0.01,
        "fp16": True,
        "bf16": False,
        "gradient_checkpointing": True,
        "max_seq_length": 2048,
        "logging_steps": 10,
        "save_strategy": "steps",
        "save_steps": 500,
        "eval_strategy": "no",
        "eval_steps": 500,
        "save_total_limit": 1,
        "dataloader_num_workers": 2,
        "per_device_eval_batch_size": 1,
        "seed": 42,
    },
}

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print("Configs loaded for:", list(CONFIGS.keys()))

## 4. Imports & Dataset Class

In [ ]:
import gc
import re
import time
import numpy as np
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, prepare_model_for_kbit_training
from tqdm.auto import tqdm


class VLCodeDataset(Dataset):
    """Loads JSONL conversations for Qwen2-VL fine-tuning."""

    def __init__(self, jsonl_path, processor, max_length=2048):
        self.processor = processor
        self.max_length = max_length
        self.samples = []
        with open(jsonl_path, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    self.samples.append(json.loads(line))
        print(f"  Loaded {len(self.samples)} examples from {jsonl_path}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        user_msg = sample["messages"][0]
        assistant_msg = sample["messages"][1]

        image_path = None
        text_prompt = ""
        for part in user_msg["content"]:
            if part["type"] == "image":
                image_path = part["image"].replace("file:///", "")
            elif part["type"] == "text":
                text_prompt = part["text"]

        target_code = assistant_msg["content"]
        image = Image.open(image_path).convert("RGB")

        conversation = [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": text_prompt},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": target_code},
            ]},
        ]

        text = self.processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        inputs = self.processor(text=[text], images=[image], padding=False, return_tensors="pt")

        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)

        if input_ids.shape[0] > self.max_length:
            input_ids = input_ids[:self.max_length]
            attention_mask = attention_mask[:self.max_length]

        labels = input_ids.clone()
        prompt_text = self.processor.apply_chat_template([conversation[0]], tokenize=False, add_generation_prompt=True)
        prompt_tokens = self.processor(text=[prompt_text], images=[image], padding=False, return_tensors="pt")
        prompt_len = min(prompt_tokens["input_ids"].shape[1], input_ids.shape[0])
        labels[:prompt_len] = -100

        result = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
        if "pixel_values" in inputs:
            result["pixel_values"] = inputs["pixel_values"].squeeze(0)
        if "image_grid_thw" in inputs:
            result["image_grid_thw"] = inputs["image_grid_thw"].squeeze(0)

        image.close()
        return result


class VLDataCollator:
    """Dynamic padding collator for variable-length sequences."""
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        max_len = max(f["input_ids"].shape[0] for f in features)
        ids, masks, labs = [], [], []
        for f in features:
            pad = max_len - f["input_ids"].shape[0]
            if pad > 0:
                ids.append(torch.cat([f["input_ids"], torch.full((pad,), self.pad_token_id, dtype=f["input_ids"].dtype)]))
                masks.append(torch.cat([f["attention_mask"], torch.zeros(pad, dtype=f["attention_mask"].dtype)]))
                labs.append(torch.cat([f["labels"], torch.full((pad,), -100, dtype=f["labels"].dtype)]))
            else:
                ids.append(f["input_ids"])
                masks.append(f["attention_mask"])
                labs.append(f["labels"])

        batch = {"input_ids": torch.stack(ids), "attention_mask": torch.stack(masks), "labels": torch.stack(labs)}
        if "pixel_values" in features[0]:
            batch["pixel_values"] = torch.cat([f["pixel_values"].unsqueeze(0) for f in features], dim=0)
        if "image_grid_thw" in features[0]:
            batch["image_grid_thw"] = torch.stack([f["image_grid_thw"] for f in features])
        return batch


print("Dataset and collator classes defined.")

## 5. Training Function

In [ ]:
def load_model_for_eval_kaggle(config, adapter_dir):
    """Load base model + adapter for eval only (saves VRAM)."""
    quant_config = None
    if config.get("quantization"):
        q = config["quantization"]
        quant_config = BitsAndBytesConfig(
            load_in_4bit=q["load_in_4bit"],
            bnb_4bit_quant_type=q["bnb_4bit_quant_type"],
            bnb_4bit_use_double_quant=q["bnb_4bit_use_double_quant"],
            bnb_4bit_compute_dtype=getattr(torch, q["bnb_4bit_compute_dtype"]),
        )
    model_kwargs = {"torch_dtype": torch.float16, "device_map": "auto"}
    if quant_config:
        model_kwargs["quantization_config"] = quant_config
    model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, **model_kwargs)
    if quant_config:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()
    processor = AutoProcessor.from_pretrained(adapter_dir, min_pixels=256*28*28, max_pixels=512*28*28)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    return model, processor


def run_validation_after_training_kaggle(adapter_dir, config):
    """Run one validation pass after training (avoids OOM)."""
    val_path = os.path.join(DATA_DIR, "val.jsonl")
    if not os.path.isfile(val_path):
        return None
    model, processor = load_model_for_eval_kaggle(config, adapter_dir)
    val_ds = VLCodeDataset(val_path, processor, config["max_seq_length"])
    eval_batch = config.get("per_device_eval_batch_size", 1)
    eval_args = TrainingArguments(
        output_dir=adapter_dir, per_device_eval_batch_size=eval_batch,
        fp16=config["fp16"], bf16=config["bf16"], report_to="none", remove_unused_columns=False,
    )
    trainer = Trainer(model=model, args=eval_args, eval_dataset=val_ds,
        data_collator=VLDataCollator(pad_token_id=processor.tokenizer.pad_token_id))
    result = trainer.evaluate()
    eval_loss = result.get("eval_loss")
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    return eval_loss


def train_method(method):
    """Fine-tune Qwen2-VL-2B with the specified PEFT method. Saves and resumes training/validation."""
    config = CONFIGS[method]
    output_dir = os.path.join(WORK_DIR, "outputs", method)
    results_dir = os.path.join(WORK_DIR, "results")
    adapter_dir = os.path.join(output_dir, "final_adapter")
    metrics_path = os.path.join(results_dir, f"{method}_training_metrics.json")
    val_path = os.path.join(DATA_DIR, "val.jsonl")
    has_val = os.path.isfile(val_path)
    adapter_done = os.path.isfile(os.path.join(adapter_dir, "adapter_config.json"))
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    # -- Resume: training already saved, run validation only if needed --
    if adapter_done:
        existing_metrics = {}
        if os.path.isfile(metrics_path):
            with open(metrics_path, "r") as f:
                existing_metrics = json.load(f)
        if has_val and existing_metrics.get("final_eval_loss") is None:
            print(f"\n{'='*60}\n  RESUME: Training saved. Running validation only.\n{'='*60}\n")
            eval_loss = run_validation_after_training_kaggle(adapter_dir, config)
            existing_metrics["final_eval_loss"] = eval_loss
            with open(metrics_path, "w") as f:
                json.dump(existing_metrics, f, indent=2)
            print(f"  Validation complete. Metrics: {metrics_path}\n")
            return existing_metrics
        print("\n  Training and validation already complete. Exiting.")
        return existing_metrics if existing_metrics else {"method": method, "adapter": adapter_dir}

    print(f"\n{'='*60}")
    print(f"  FINE-TUNING: {method.upper()}")
    print(f"  {config['description']}")
    print(f"{'='*60}\n")

    # -- Quantization --
    quant_config = None
    if config.get("quantization"):
        q = config["quantization"]
        quant_config = BitsAndBytesConfig(
            load_in_4bit=q["load_in_4bit"],
            bnb_4bit_quant_type=q["bnb_4bit_quant_type"],
            bnb_4bit_use_double_quant=q["bnb_4bit_use_double_quant"],
            bnb_4bit_compute_dtype=getattr(torch, q["bnb_4bit_compute_dtype"]),
        )

    gc.collect()
    torch.cuda.empty_cache()

    # -- Load model --
    model_kwargs = {"torch_dtype": torch.float16, "device_map": "auto"}
    if quant_config:
        model_kwargs["quantization_config"] = quant_config

    print(f"Loading {MODEL_ID}...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, **model_kwargs)

    if quant_config:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=512*28*28)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    # -- PEFT --
    lora_cfg = LoraConfig(
        r=config["lora_r"], lora_alpha=config["lora_alpha"], lora_dropout=config["lora_dropout"],
        target_modules=config["target_modules"], task_type=TaskType.CAUSAL_LM,
        use_dora=config.get("use_dora", False),
    )
    model = get_peft_model(model, lora_cfg)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({trainable/total:.2%})")

    free, total_mem = torch.cuda.mem_get_info()
    print(f"  GPU: {free/1024**3:.1f} GiB free / {total_mem/1024**3:.1f} GiB total")

    # -- Datasets (no in-loop eval to avoid OOM; validation runs after training) --
    print("\nLoading datasets...")
    train_ds = VLCodeDataset(os.path.join(DATA_DIR, "train.jsonl"), processor, config["max_seq_length"])
    val_ds = VLCodeDataset(os.path.join(DATA_DIR, "val.jsonl"), processor, config["max_seq_length"]) if has_val else None
    if has_val:
        print("  Validation will run once after training (saves VRAM).")

    # -- Trainer --
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=config["num_train_epochs"],
        per_device_train_batch_size=config["per_device_train_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        learning_rate=config["learning_rate"],
        warmup_steps=config["warmup_steps"],
        weight_decay=config["weight_decay"],
        fp16=config["fp16"], bf16=config["bf16"],
        gradient_checkpointing=config["gradient_checkpointing"],
        logging_steps=config["logging_steps"],
        save_strategy=config["save_strategy"],
        save_steps=config.get("save_steps", 500),
        eval_strategy=config["eval_strategy"],
        eval_steps=config.get("eval_steps", 500),
        save_total_limit=config["save_total_limit"],
        dataloader_num_workers=config["dataloader_num_workers"],
        seed=config["seed"],
        report_to="none",
        remove_unused_columns=False,
        load_best_model_at_end=False,
        optim="adamw_torch",
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=None,
        data_collator=VLDataCollator(pad_token_id=processor.tokenizer.pad_token_id),
    )

    # -- Auto-resume from checkpoint if session restarted --
    t0 = time.time()
    torch.cuda.reset_peak_memory_stats()

    last_checkpoint = None
    if os.path.isdir(output_dir):
        checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
        if checkpoints:
            last_checkpoint = os.path.join(output_dir, sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1])
            print(f'  Resuming from checkpoint: {last_checkpoint}')
        else:
            print('  Starting fresh (no checkpoints found)')

    trainer.train(resume_from_checkpoint=last_checkpoint)
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1024**3

    # -- Save adapter --
    model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)
    last_log = trainer.state.log_history[-1] if trainer.state.log_history else {}

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    # -- Run validation after training (avoids OOM) --
    final_eval_loss = None
    if has_val:
        final_eval_loss = run_validation_after_training_kaggle(adapter_dir, config)

    metrics = {
        "method": method, "trainable_params": trainable, "total_params": total,
        "trainable_pct": trainable / total, "peak_vram_gb": round(peak_mem, 2),
        "training_time_hours": round(elapsed / 3600, 3),
        "training_time_seconds": round(elapsed, 1),
        "train_samples": len(train_ds), "val_samples": len(val_ds) if val_ds else 0,
        "final_train_loss": last_log.get("train_loss"),
        "final_eval_loss": final_eval_loss if final_eval_loss is not None else last_log.get("eval_loss"),
        "quantized": config.get("quantization") is not None,
        "use_dora": config.get("use_dora", False),
    }
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"\n{'='*60}")
    print(f"  DONE: {method.upper()} in {elapsed/3600:.2f} hrs | Peak VRAM: {peak_mem:.2f} GiB")
    print(f"  Adapter: {adapter_dir}")
    print(f"{'='*60}")

    return metrics


print("Training function defined.")

## 6. Evaluation Function

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
from rouge_score import rouge_scorer
from codebleu import calc_codebleu


def strip_code_fences(text):
    s = text.strip()
    m = re.search(r'```(?:\w+)?\s*\n(.*?)```', s, re.DOTALL)
    if m:
        return m.group(1).strip()
    if s.startswith('```'):
        s = re.sub(r'^```\w*\n?', '', s)
        s = re.sub(r'\n?```$', '', s)
        return s.strip()
    return s


def validate_jsx_python(code):
    """Python-based JSX validation (no Node.js needed)."""
    if not code.strip():
        return False, "empty"
    if "GeneratedComponent" not in code and "export" not in code:
        return False, "missing component export"
    opens = code.count("<") - code.count("</") - code.count("/>")
    if opens < -5:
        return False, "severely unbalanced tags"
    if code.count("(") != code.count(")"):
        return False, "unbalanced parentheses"
    if code.count("{") != code.count("}"):
        return False, "unbalanced braces"
    return True, ""


def validate_vue_python(code):
    """Python-based Vue SFC validation."""
    if not code.strip():
        return False, "empty"
    if "<template>" not in code:
        return False, "missing <template>"
    if "</template>" not in code:
        return False, "missing </template>"
    return True, ""


def validate_html_python(code):
    if not code.strip():
        return False, "empty"
    if "<" not in code:
        return False, "no HTML tags"
    return True, ""


VALIDATORS = {"html": validate_html_python, "jsx": validate_jsx_python, "vue": validate_vue_python}
TASK_KEYWORDS = {
    "jsx": ["React JSX", "GeneratedComponent"],
    "vue": ["Vue 3", "Single File Component"],
    "html": ["HTML"],
}


def detect_task(prompt):
    for task, kws in TASK_KEYWORDS.items():
        for kw in kws:
            if kw in prompt:
                return task
    return "html"


def compute_code_metrics(preds, refs, lang="javascript"):
    bleu = BLEU(effective_order=True)
    chrf = CHRF()
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

    bleu_score = bleu.corpus_score(preds, [refs])
    chrf_score = chrf.corpus_score(preds, [refs])
    rouge_scores = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]

    try:
        cb = calc_codebleu(references=refs, predictions=preds, lang=lang, weights=(0.25, 0.25, 0.25, 0.25))
        codebleu = cb["codebleu"]
    except Exception:
        codebleu = 0.0

    return {
        "bleu4": round(bleu_score.score, 4),
        "chrf": round(chrf_score.score, 4),
        "rouge_l": round(float(np.mean(rouge_scores)), 4) if rouge_scores else 0.0,
        "codebleu": round(codebleu, 4),
    }


def evaluate_method(method, max_samples=None):
    """Run inference on test set and compute all code metrics."""
    results_dir = os.path.join(WORK_DIR, "results")
    os.makedirs(results_dir, exist_ok=True)

    test_path = os.path.join(DATA_DIR, "test.jsonl")
    test_samples = []
    with open(test_path, "r") as f:
        for line in f:
            if line.strip():
                test_samples.append(json.loads(line))
    if max_samples:
        test_samples = test_samples[:max_samples]

    print(f"\nEvaluating {method.upper()} on {len(test_samples)} test samples...")

    gc.collect()
    torch.cuda.empty_cache()

    is_qlora = method == "qlora"
    model_kwargs = {"torch_dtype": torch.float16, "device_map": "auto"}
    if is_qlora:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
        )

    model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, **model_kwargs)
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=512*28*28)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    if method != "base":
        adapter_path = os.path.join(WORK_DIR, "outputs", method, "final_adapter")
        print(f"  Loading adapter: {adapter_path}")
        model = PeftModel.from_pretrained(model, adapter_path)

    model.eval()

    results = {"html": [], "jsx": [], "vue": []}
    latencies = []

    for sample in tqdm(test_samples, desc="Inference"):
        user_msg = sample["messages"][0]
        reference = sample["messages"][1]["content"]

        image_path, text_prompt = None, ""
        for part in user_msg["content"]:
            if part["type"] == "image":
                image_path = part["image"].replace("file:///", "")
            elif part["type"] == "text":
                text_prompt = part["text"]

        task = detect_task(text_prompt)
        image = Image.open(image_path).convert("RGB")

        conversation = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": text_prompt},
        ]}]
        text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True).to(model.device)

        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=2048, do_sample=False, pad_token_id=processor.tokenizer.pad_token_id)
        latencies.append(time.time() - t0)

        pred = processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = strip_code_fences(pred)
        ok, err = VALIDATORS[task](pred)

        results[task].append({"prediction": pred, "reference": reference, "parse_ok": ok})
        image.close()
        del inputs, out
        torch.cuda.empty_cache()

    # Compute metrics per task
    all_metrics = {}
    for task in ["html", "jsx", "vue"]:
        entries = results[task]
        if not entries:
            continue
        preds = [e["prediction"] for e in entries]
        refs = [e["reference"] for e in entries]
        psr = sum(1 for e in entries if e["parse_ok"]) / len(entries)
        code_m = compute_code_metrics(preds, refs)
        all_metrics[task] = {"n_samples": len(entries), "parse_success_rate": round(psr, 4), **code_m}
        print(f"  {task.upper()}: PSR={psr:.2%} BLEU={code_m['bleu4']:.4f} CodeBLEU={code_m['codebleu']:.4f}")

    total_n = sum(m["n_samples"] for m in all_metrics.values())
    summary = {
        "method": method, "total_samples": total_n,
        "avg_inference_latency_s": round(float(np.mean(latencies)), 3),
        "per_task": all_metrics,
    }
    for metric in ["parse_success_rate", "bleu4", "codebleu", "rouge_l", "chrf"]:
        summary[f"avg_{metric}"] = round(
            sum(m[metric] * m["n_samples"] for m in all_metrics.values()) / total_n, 4
        ) if total_n else 0

    with open(os.path.join(results_dir, f"{method}_eval_results.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n  Avg PSR: {summary.get('avg_parse_success_rate', 0):.2%}")
    print(f"  Avg BLEU-4: {summary.get('avg_bleu4', 0):.4f}")
    print(f"  Avg Latency: {summary.get('avg_inference_latency_s', 0):.2f}s")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return summary


print("Evaluation function defined.")

## 6b. Restore from previous session (to resume training)

**Only run this if you stopped a session and need to resume from saved checkpoints.**

**Upload properly:**
1. In a previous run, use the **Download** cell (section 10) to create `finetune_results.zip` in Output, then download it.
2. Go to **kaggle.com** → **Your work** → **Datasets** → **New dataset**.
3. Upload **only** the `finetune_results.zip` file (do not zip it again). Give the dataset any name (e.g. "finetune-results").
4. In this notebook, click **Add Data** (right sidebar), search for your dataset, and add it.
5. Run the cell below. It will list folders under `/kaggle/input/` — your restore dataset is the one that is *not* finetune1_data or websight-images. You can set `RESTORE_FROM` to that exact name, or leave it empty to auto-detect.

In [ ]:
# Set to the exact folder name under /kaggle/input/ (see list below). Leave "" to auto-detect a folder with a .zip or outputs/results.
RESTORE_FROM = ""

import shutil
import zipfile

# List all input folders (Kaggle may use 'datasets' as parent; datasets can be nested)
def list_input_tree(path, depth=0, max_depth=2):
    seen = []
    if depth < max_depth and os.path.isdir(path):
        for d in sorted(os.listdir(path)):
            p = os.path.join(path, d)
            if os.path.isdir(p): seen.append(p.replace(KAGGLE_INPUT, '').lstrip('/'))
    return seen
input_folders = [d for d in os.listdir(KAGGLE_INPUT) if os.path.isdir(os.path.join(KAGGLE_INPUT, d))]
all_candidates = list(input_folders)
for f in input_folders:
    sub = os.path.join(KAGGLE_INPUT, f)
    if os.path.isdir(sub):
        for d in os.listdir(sub):
            fp = os.path.join(sub, d)
            if os.path.isdir(fp):
                all_candidates.append(os.path.join(f, d))
                for d2 in os.listdir(fp):
                    fp2 = os.path.join(fp, d2)
                    if os.path.isdir(fp2): all_candidates.append(os.path.join(f, d, d2))
print("Paths to try:", all_candidates)

def try_restore_from(src_dir):
    if not os.path.isdir(src_dir):
        return False
    zip_path = None
    for f in os.listdir(src_dir):
        if f.endswith(".zip"):
            zip_path = os.path.join(src_dir, f)
            break
    if zip_path and os.path.isfile(zip_path):
        extract_to = os.path.join(WORK_DIR, "_restore_tmp")
        os.makedirs(extract_to, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_to)
        got_outputs = False
        for name in ["outputs", "results"]:
            src_sub = os.path.join(extract_to, name)
            dst_sub = os.path.join(WORK_DIR, name)
            if os.path.isdir(src_sub):
                shutil.copytree(src_sub, dst_sub, dirs_exist_ok=True)
                print(f"  Restored {name}/ -> {dst_sub}")
                if name == "outputs": got_outputs = True
        shutil.rmtree(extract_to, ignore_errors=True)
        return got_outputs
    out_dir = os.path.join(src_dir, "outputs")
    if os.path.isdir(out_dir):
        shutil.copytree(out_dir, os.path.join(WORK_DIR, "outputs"), dirs_exist_ok=True)
        print(f"  Restored outputs/ -> {os.path.join(WORK_DIR, 'outputs')}")
        res_dir = os.path.join(src_dir, "results")
        if os.path.isdir(res_dir):
            shutil.copytree(res_dir, os.path.join(WORK_DIR, "results"), dirs_exist_ok=True)
            print(f"  Restored results/ -> {os.path.join(WORK_DIR, 'results')}")
        return True
    return False

restored = False
skip_names = ("finetune1_data", "websight-images", "images")
candidates = [c for c in all_candidates if not any(s in c for s in skip_names)]
if RESTORE_FROM:
    candidates = [RESTORE_FROM.strip()] + [c for c in candidates if c != RESTORE_FROM.strip()]
for rel in candidates:
    parts = rel.replace("\\", "/").split("/")
    src = os.path.join(KAGGLE_INPUT, *parts)
    if try_restore_from(src):
        restored = True
        print(f"  Restored from: {rel}")
        print("  Restore done. Run the 'Run Training' cell to resume.")
        break
if not restored:
    print("  No restore dataset found. Set RESTORE_FROM to a path above (e.g. datasets/finetune_results).")

## 7. Run Training

**Run ONE method per Kaggle session** (12hr limit). Change `METHOD` below.

**To resume after stopping:** Run the **Restore** cell (6b) first (after adding your saved-outputs dataset), then run this cell. Checkpoints are saved every 500 steps; without saving/restoring outputs, a new session cannot resume.

In [ ]:
# === CHANGE THIS FOR EACH SESSION ===
METHOD = "qlora"  # Options: "lora", "qlora", "dora"
# ====================================

train_metrics = train_method(METHOD)
print(json.dumps(train_metrics, indent=2))

## 8. Run Evaluation

Evaluate the method you just trained. Use `max_samples` to test quickly first.

In [ ]:
eval_results = evaluate_method(METHOD, max_samples=None)
print(json.dumps(eval_results, indent=2))

## 9. Compare Results

Run this cell after all three methods have been trained and evaluated.
Upload the `results/` JSON files from previous sessions if running in a new session.

In [ ]:
import matplotlib.pyplot as plt

results_dir = os.path.join(WORK_DIR, "results")
METHODS = ["lora", "qlora", "dora"]
LABELS = {"lora": "LoRA", "qlora": "QLoRA", "dora": "DoRA"}
COLORS = {"lora": "#4C78A8", "qlora": "#F58518", "dora": "#54A24B"}

data = {}
for m in METHODS:
    tp = os.path.join(results_dir, f"{m}_training_metrics.json")
    ep = os.path.join(results_dir, f"{m}_eval_results.json")
    entry = {}
    if os.path.exists(tp):
        with open(tp) as f: entry["training"] = json.load(f)
    if os.path.exists(ep):
        with open(ep) as f: entry["eval"] = json.load(f)
    if entry:
        data[m] = entry

found = [m for m in METHODS if m in data]
print(f"Found results for: {', '.join(LABELS[m] for m in found)}")

if not found:
    print("No results found. Train and evaluate at least one method first.")
else:
    # -- Table --
    print(f"\n{'Metric':<28} {'LoRA':>12} {'QLoRA':>12} {'DoRA':>12}")
    print("-" * 70)
    rows = [
        ("Trainable %", "training", "trainable_pct", lambda v: f"{v:.2%}"),
        ("Peak VRAM (GiB)", "training", "peak_vram_gb", lambda v: f"{v:.2f}"),
        ("Training Time (hrs)", "training", "training_time_hours", lambda v: f"{v:.2f}"),
        ("Avg PSR", "eval", "avg_parse_success_rate", lambda v: f"{v:.2%}"),
        ("Avg BLEU-4", "eval", "avg_bleu4", lambda v: f"{v:.4f}"),
        ("Avg CodeBLEU", "eval", "avg_codebleu", lambda v: f"{v:.4f}"),
        ("Avg ROUGE-L", "eval", "avg_rouge_l", lambda v: f"{v:.4f}"),
        ("Avg ChrF", "eval", "avg_chrf", lambda v: f"{v:.4f}"),
        ("Latency (s)", "eval", "avg_inference_latency_s", lambda v: f"{v:.2f}"),
    ]
    for label, sec, key, fmt in rows:
        vals = []
        for m in METHODS:
            v = data.get(m, {}).get(sec, {}).get(key)
            vals.append(fmt(v) if v is not None else "N/A")
        print(f"  {label:<26} {vals[0]:>12} {vals[1]:>12} {vals[2]:>12}")

    # -- Bar chart --
    chart_metrics = [
        ("PSR", "eval", "avg_parse_success_rate"),
        ("BLEU-4", "eval", "avg_bleu4"),
        ("CodeBLEU", "eval", "avg_codebleu"),
        ("ROUGE-L", "eval", "avg_rouge_l"),
        ("ChrF", "eval", "avg_chrf"),
    ]
    fig, axes = plt.subplots(1, len(chart_metrics), figsize=(18, 4))
    for ax, (title, sec, key) in zip(axes, chart_metrics):
        vals, names, cols = [], [], []
        for m in found:
            v = data[m].get(sec, {}).get(key, 0) or 0
            vals.append(v)
            names.append(LABELS[m])
            cols.append(COLORS[m])
        bars = ax.bar(names, vals, color=cols)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, b.get_height(), f"{v:.4f}", ha="center", va="bottom", fontsize=8)
        ax.set_title(title, fontweight="bold")
        ax.set_ylim(0, max(vals)*1.3 if vals and max(vals) > 0 else 1)
    plt.suptitle("LoRA vs QLoRA vs DoRA", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "comparison_chart.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nChart saved to {results_dir}/comparison_chart.png")

## 10. Download Results

Zip all results and adapters for download back to your local machine.

In [ ]:
import shutil

zip_path = os.path.join(WORK_DIR, "finetune_results")

# Remove old zip if it exists to reclaim space
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")
    print("Removed old finetune_results.zip")

# Collect files to zip
collect_dir = os.path.join(WORK_DIR, "_to_zip")
if os.path.exists(collect_dir):
    shutil.rmtree(collect_dir)
os.makedirs(collect_dir, exist_ok=True)

# Copy results
results_src = os.path.join(WORK_DIR, "results")
if os.path.exists(results_src):
    shutil.copytree(results_src, os.path.join(collect_dir, "results"), dirs_exist_ok=True)

# Copy outputs -- only final_adapter + latest checkpoint per method (saves space)
outputs_src = os.path.join(WORK_DIR, "outputs")
if os.path.exists(outputs_src):
    for method_name in os.listdir(outputs_src):
        method_dir = os.path.join(outputs_src, method_name)
        if not os.path.isdir(method_dir):
            continue
        dst_method = os.path.join(collect_dir, "outputs", method_name)
        os.makedirs(dst_method, exist_ok=True)

        # Copy final_adapter if it exists
        adapter = os.path.join(method_dir, "final_adapter")
        if os.path.isdir(adapter):
            shutil.copytree(adapter, os.path.join(dst_method, "final_adapter"))

        # Copy only the latest checkpoint (for resume)
        checkpoints = sorted(
            [d for d in os.listdir(method_dir) if d.startswith("checkpoint-")],
            key=lambda x: int(x.split("-")[1]),
        )
        if checkpoints:
            latest = checkpoints[-1]
            shutil.copytree(
                os.path.join(method_dir, latest),
                os.path.join(dst_method, latest),
            )
            print(f"  {method_name}: packed {latest}" + (" + final_adapter" if os.path.isdir(adapter) else ""))
        elif os.path.isdir(adapter):
            print(f"  {method_name}: packed final_adapter (training complete, no checkpoints)")

        # Copy any loose files (e.g. trainer_state.json, training_args.bin)
        for fname in os.listdir(method_dir):
            fpath = os.path.join(method_dir, fname)
            if os.path.isfile(fpath):
                shutil.copy2(fpath, os.path.join(dst_method, fname))

shutil.make_archive(zip_path, "zip", collect_dir)
shutil.rmtree(collect_dir)

print(f"\nDownload: {zip_path}.zip ({os.path.getsize(zip_path + '.zip') / 1024 / 1024:.1f} MB)")
print("Go to: /kaggle/working/finetune_results.zip -> click to download")